In [ ]:
!pip install --upgrade -q transformers torchtext

In [2]:
import torch
import torchvision
from torch.utils.data import DataLoader,Dataset
import torch.nn as nn
import torch.optim as optim
import os
from tqdm import tqdm
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aftaab/mscoco")

print("Path to dataset files:", path)

In [ ]:


dataset_root = "/root/.cache/kagglehub/datasets/aftaab/mscoco/versions/1"

print(f"Listing contents of the dataset directory: {dataset_root}")
print("--------------------------------------------------")

for root, dirs, files in os.walk(dataset_root):
    # Print current directory
    print(f"Directory: {root}")
    # Print subdirectories
    if dirs:
        print(f"  Subdirectories: {', '.join(dirs)}")
    # Print files in current directory
    if files:
        print(f"  Files: {', '.join(files)}")
    print("--------------------------------------------------")

My approach begins with Hugging Face pipelines, which I will use to automatically generate textual descriptions for a large set of images. This step is crucial because it allows me to build my own image–report dataset: each image will be paired with a generated explanation that serves as training data.

Once this dataset is prepared, I will train two interconnected models:

1 . Image Analysis Model


2 . Report Generation Model

By following this pipeline, Hugging Face is not the end goal but rather the first step: it helps me bootstrap a dataset of image–report pairs. With that dataset, I can train my own specialized models where one handles image analysis and the other handles report generation, working together to produce detailed explanations of uploaded images.

In [ ]:
imgs = []

for dir_item in os.listdir(path):
  if dir_item == "mscoco_resized":
    mscoco_resized_path = os.path.join(path, dir_item)
    # Check if mscoco_resized_path is indeed a directory before listing
    if os.path.isdir(mscoco_resized_path):
      for sub_dir in os.listdir(mscoco_resized_path):
        current_image_dir = os.path.join(mscoco_resized_path, sub_dir)
        # Check if current_image_dir is indeed a directory before listing its contents
        if os.path.isdir(current_image_dir):
          for img_name in os.listdir(current_image_dir):
            if img_name.endswith(".jpg"):
              imgs.append(os.path.join(current_image_dir, img_name)) # Append full path to image

print(f"Found {len(imgs)} image files.")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torch 

class ImageDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
      return len(self.image_paths)

    def __getitem__(self, idx):
       img_path = self.image_paths[idx]
       image = Image.open(img_path).convert("RGB")
       if self.transform:
           image = self.transform(image)
       return image, img_path # Return both the image tensor and its path


class CombinedImageTextDataset(Dataset):
    def __init__(self, image_paths, img_report_tokenized_dict, tokenizer, transform=None, max_seq_len=30):
        self.image_paths = image_paths
        self.img_report_tokenized_dict = img_report_tokenized_dict
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_seq_len = max_seq_len
        # Get the pad_token_id once
        self.pad_token_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else 0

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        tokenized_caption_words = self.img_report_tokenized_dict.get(img_path, [])
        caption_ids = self.tokenizer.convert_tokens_to_ids(tokenized_caption_words)

        if not caption_ids: # Handle empty caption case with pad_token_id
            caption_ids = [self.pad_token_id]

        if len(caption_ids) < self.max_seq_len:
            padding_needed = self.max_seq_len - len(caption_ids)
            caption_ids = caption_ids + [self.pad_token_id] * padding_needed
        elif len(caption_ids) > self.max_seq_len:
            caption_ids = caption_ids[:self.max_seq_len]

        caption_ids_tensor = torch.tensor(caption_ids, dtype=torch.long)

        return image, caption_ids_tensor

# Define the transformation to convert PIL Images to PyTorch Tensors
transform = transforms.ToTensor()

# Re-create the dataset with the transform (this `dataset` instance is used by ModelVision for shape inference)
dataset = ImageDataset(imgs, transform=transform)



print(f"Found {len(imgs)} image files.")
print(f"Shape of a single transformed image: {dataset[0][0].shape}") # Access image from the (image, path) tuple


In [ ]:

if 'processor' not in locals() or 'model' not in locals():
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to("cuda") # Move model to GPU


if 'imgs_reports' not in locals():
    imgs_reports = {}

# Move model to GPU if available
if torch.cuda.is_available():
    model.to("cuda")

if 'transform' not in locals():
    transform = transforms.ToTensor()

# Create an ImageDataset and DataLoader specifically for caption generation
caption_dataset = ImageDataset(imgs, transform=transform)
caption_data_loader = DataLoader(caption_dataset, batch_size=32, shuffle=False, num_workers=os.cpu_count())


# Continue generating reports for all images
# Using tqdm for a progress bar
for batch_idx, (image_batch, paths_batch) in enumerate(tqdm(caption_data_loader, desc="Generating image reports")):
    # Move image batch to GPU
    if torch.cuda.is_available():
        image_batch = image_batch.to("cuda")

    inputs = processor(images=image_batch, return_tensors="pt").to("cuda")

    # Generate captions for the batch, explicitly setting max_new_tokens
    out_ids = model.generate(**inputs, max_new_tokens=30)
    captions = processor.batch_decode(out_ids, skip_special_tokens=True)


    for img_path, caption in zip(paths_batch, captions):
        if img_path not in imgs_reports:
            imgs_reports[img_path] = caption

print(f"Generated reports for {len(imgs_reports)} images.")
for i, (img_path, report) in enumerate(list(imgs_reports.items())[:5]):
  print(f"Image: {img_path}\nReport: {report}\n")


In [ ]:
img_report_tokenized = {}
blip_tokenizer = processor.tokenizer

for path, caption in imgs_reports.items():
  # Tokenize the caption into a list of words using the BLIP tokenizer
  img_report_tokenized[path] = blip_tokenizer.tokenize(caption)

print(f"Tokenized reports for {len(img_report_tokenized)} images.")
print("First 5 tokenized reports:")
for i, (img_path, tokens) in enumerate(list(img_report_tokenized.items())[:5]):
  print(f"Image: {img_path}\nTokens: {tokens}\n")

In [7]:
class ModelVision(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv_1  = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
    self.conv_2  = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
    self.conv_3  = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
    self.conv_4  = nn.Conv2d(in_channels=128, out_channels=64, kernel_size=3, padding=1)
    self.conv_5  = nn.Conv2d(in_channels=64, out_channels=32, kernel_size=3, padding=1)
    self.pool = nn.AdaptiveAvgPool2d((8, 8))
    self.linear  = nn.Linear(in_features=32 * 8 * 8, out_features=64)

  def forward(self, x):
    x = self.conv_5(self.conv_4(self.conv_3(self.conv_2(self.conv_1(x)))))
    x = self.pool(x)
    # Flatten the tensor before passing to the linear layer
    x = x.view(x.size(0), -1)
    x = self.linear(x)
    return x

class ModelText(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embed = nn.Embedding(num_embeddings=vocab_size, embedding_dim=64)
    self.query_proj = nn.Linear(in_features=64, out_features=64)
    self.key_proj = nn.Linear(in_features=64, out_features=64)
    self.value_proj = nn.Linear(in_features=64, out_features=64)
    self.maskAtt = nn.MultiheadAttention(embed_dim=64, num_heads=8, batch_first=True)
    self.add_norm_1 = nn.LayerNorm(64)
    self.cross_attention = nn.MultiheadAttention(embed_dim=64, num_heads=8, batch_first=True)
    self.add_norm_2 = nn.LayerNorm(64)
    self.linear = nn.Linear(in_features=64, out_features=vocab_size)


  def forward(self, text_input_ids, memory_vision):

    embedded_text = self.embed(text_input_ids) # (batch_size, sequence_length, embed_dim)
    query_text = self.query_proj(embedded_text)
    key_text = self.key_proj(embedded_text)
    value_text = self.value_proj(embedded_text)

    self_attention_output, _ = self.maskAtt(query=query_text, key=key_text, value=value_text)
    attention_output_norm = self.add_norm_1(self_attention_output + embedded_text)
    memory_vision_expanded = memory_vision.unsqueeze(1) # (batch_size, 1, embed_dim)

    cross_attention_output, _ = self.cross_attention(query=attention_output_norm,
                                                     key=memory_vision_expanded,
                                                     value=memory_vision_expanded)

    output = self.add_norm_2(cross_attention_output + attention_output_norm)
    output = self.linear(output)
    return output

In [ ]:
EPOCHS = 5 # i will use 100 epochs but Reduced to 5 for quicker testing
LEARNING_RATE = 0.001

model_vision = ModelVision()
model_text = ModelText(vocab_size=blip_tokenizer.vocab_size)

# Move models to GPU if available
if torch.cuda.is_available():
    model_vision.to("cuda")
    model_text.to("cuda")

# Create the combined dataset for training
combined_dataset_for_training = CombinedImageTextDataset(
    image_paths=imgs,
    img_report_tokenized_dict=img_report_tokenized,
    tokenizer=blip_tokenizer,
    transform=transform,
    max_seq_len=30 # Using a fixed length for captions
)


data_loader = DataLoader(combined_dataset_for_training, batch_size=32, pin_memory=True, num_workers=os.cpu_count())

optimizer_vision = optim.Adam(model_vision.parameters(), lr=LEARNING_RATE)
optimizer_text = optim.Adam(model_text.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=blip_tokenizer.pad_token_id)
loss_value = []

for e in tqdm(range(EPOCHS)):
    for batch_idx, (image_batch, captions_batch) in enumerate(data_loader):
        model_vision.train()
        model_text.train()


        if torch.cuda.is_available():
            image_batch = image_batch.to("cuda")
            captions_batch = captions_batch.to("cuda")

        output_vision = model_vision(image_batch)
        # Ensure ModelText receives the correct input shape (batch_size, sequence_length) for captions
        output_text = model_text(captions_batch , output_vision)

        loss = criterion(output_text.view(-1, output_text.size(-1)), captions_batch.view(-1)) # Reshape for CrossEntropyLoss
        loss.backward()

        optimizer_vision.zero_grad()
        optimizer_text.zero_grad()

        optimizer_vision.step()
        optimizer_text.step()

        loss_value.append(loss.item())

    loss_per_epoch = sum(loss_value) / len(loss_value) # Calculate average loss for the epoch
    # Print loss for every epoch for better monitoring
    print(f"\nEpoch: {e}, Loss: {loss_per_epoch:.2f}%")
    print("----------------------------------------------------------------")
    loss_value = [] # Reset loss_value for the next epoch

In [ ]:
from google.colab import files
import cv2


In [ ]:
def EVRG(img):
  img_to_tensor = transforms.ToTensor()(img)
  img_to_tensor = img_to_tensor.unsqueeze(0)
  model_vision.eval()
  with torch.no_grad():
    out =  model_vision(img_to_tensor)
    cap =  model_text(out)
    word = blip_tokenizer.decode(cap.argmax(dim=-1)[0])
    return word

In [ ]:
print("UPLOAD YOUR IMAGE FROM YOUR LOCAL MACHINE TO GET REPORT ABOUT IT")
image = None
if not os.path.exist(image):
  print("click on button upload ")
  try :
      image = files.upload()
      print("the image uploaded successfully")
  except :
      print("something went wrong")
else:
  print("the image already exists")

print("-------------------------------------------------------------------------")
img = cv2.imread(image)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
cv2.imshow("image :",img)
print("---------------------------------")
print("caption of image generated by model EVRG : \n")
for it in range(50):
  word = EVRG(image)
  print("".join(word))